# SCOS Analysis — Interactive Results Explorer

Browse the results produced by `scos_analysis.py`: pick a measurement (a `SCOS_Scan_*` offset or a
`ref-*/SCOS_Single_*` PaLS-iSCOS recording), a metric (`raw` / `corrected` / `BFi` / `intensity`), and
a smoothing filter to apply live. Shows the time-domain signal, its FFT/PSD, and the SNR/BPM values
from `scos_analysis_results.csv` alongside a live-recomputed SNR for the filtered signal.

**Note on file layout — this is different from the autocorr pipeline's explorer:**
`scos_analysis.py` saves one `.npy` file per *metric* per folder (`time_vector.npy`, `K2_raw.npy`,
`K2_corrected.npy`, `BFi.npy`, `intensity.npy`), not one file per offset:
- For a `SCOS_Scan_*` folder (Gated SCOS), each of those files holds an **object array**, one entry
  per gate offset — so offset `i`'s signal is `K2_raw.npy[i]`, not a separate file.
- For a `ref-*/SCOS_Single_*` folder (PaLS-iSCOS), each file is just a **single plain array** for
  that one measurement.

The helpers below resolve both cases transparently.

In [ ]:
# 1. Imports & base path selection
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display, clear_output, Image
import ipywidgets as widgets
from ipywidgets import Dropdown, Checkbox, Layout, HBox, VBox, Output
from scipy.signal import butter, filtfilt, savgol_filter
from scipy.ndimage import uniform_filter1d, median_filter

# --- Option A: pick the folder with a file dialog (uncomment to use) ---
# import tkinter as tk
# from tkinter import filedialog
# def select_folder_dialog():
#     root = tk.Tk()
#     root.withdraw()
#     folder = filedialog.askdirectory(title='Select the base path used for scos_analysis.py')
#     root.destroy()
#     return folder
# BASE_PATH = select_folder_dialog()

# --- Option B: set the base path manually (the same folder you passed to scos_analysis.py) ---
BASE_PATH = r"E:\Shira_TZ_ISCOS\19-04-2026 Gated and PaLS-iSCOS 1cmSDS 10bit with Pileup"

BASE_PATH = Path(BASE_PATH)
CSV_PATH = BASE_PATH / 'scos_analysis_results.csv'

if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {CSV_PATH}. Make sure BASE_PATH points to the same folder you passed "
        f"to scos_analysis.py, and that the pipeline has already been run there."
    )

results_df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(results_df)} rows from {CSV_PATH}")
display(results_df.head())

Loaded 405 rows from E:\Shira_TZ_ISCOS\26-04-2026 1cm Gated +ISCOS\scos_analysis_results.csv


,folder,gate_offset,gate_index,relative_offset_ns,method,metric,SNR_dB,BPM,f_max_Hz
0,SCOS_Scan_1777205818-40%,40384.2,0.0,-300.0,Gated SCOS,raw,22.803478,87.090671,1.451511
1,SCOS_Scan_1777205818-40%,40384.2,0.0,-300.0,Gated SCOS,corrected,22.990884,87.090671,1.451511
2,SCOS_Scan_1777205818-40%,40384.2,0.0,-300.0,Gated SCOS,intensity,20.531195,87.090671,1.451511
3,SCOS_Scan_1777205818-40%,40484.2,1.0,-200.0,Gated SCOS,raw,22.113192,87.090671,1.451511
4,SCOS_Scan_1777205818-40%,40484.2,1.0,-200.0,Gated SCOS,corrected,23.864800,87.090671,1.451511


In [2]:
# 2. Helpers: resolve where a measurement's .npy files live, and load them
#
# Reverse of SCOSAnalyzer.metric_map: CSV 'metric' labels -> the .npy filename (without extension)
# that scos_analysis.py actually saved under.
REVERSE_METRIC_MAP = {'raw': 'K2_raw', 'corrected': 'K2_corrected', 'BFi': 'BFi', 'intensity': 'intensity'}
METRIC_YLABELS = {'raw': 'K2_raw', 'corrected': 'K2_corrected', 'BFi': 'BFi', 'intensity': 'Intensity'}


def resolve_scos_dir(base_path, folder):
    """
    scos_analysis.py stores 'folder' in the CSV as either:
      - 'SCOS_Scan_X'                (Gated SCOS -> base_path/SCOS_Scan_X)
      - 'ref-Y/SCOS_Single_Z'         (PaLS-iSCOS -> base_path/ref-Y/SCOS_Single_Z)
    Returns (directory, is_scan) where is_scan tells us whether the .npy files in that
    directory are object arrays indexed by gate_index (True) or plain single arrays (False).
    """
    base_path = Path(base_path)
    folder = str(folder)
    if '/' in folder:
        ref_name, single_name = folder.split('/', 1)
        return base_path / ref_name / single_name, False
    return base_path / folder, True


def load_scos_measurement(base_path, folder, gate_index):
    """
    Load (time_vector, {metric_label: signal_or_None}) for one measurement, transparently
    handling both the per-offset object-array layout (SCOS_Scan_*) and the plain-array
    layout (SCOS_Single_*).
    """
    d, is_scan = resolve_scos_dir(base_path, folder)

    all_time = np.load(d / 'time_vector.npy', allow_pickle=True)
    raw_signals = {}
    for label, fname in REVERSE_METRIC_MAP.items():
        fpath = d / f"{fname}.npy"
        raw_signals[label] = np.load(fpath, allow_pickle=True) if fpath.exists() else None

    if is_scan and pd.notna(gate_index):
        i = int(gate_index)
        time_vec = np.asarray(all_time[i]).flatten()
        signals = {
            label: (np.asarray(arr[i]).flatten() if arr is not None and arr[i] is not None else None)
            for label, arr in raw_signals.items()
        }
    else:
        time_vec = np.asarray(all_time).flatten()
        signals = {
            label: (np.asarray(arr).flatten() if arr is not None else None)
            for label, arr in raw_signals.items()
        }
    return time_vec, signals


def scos_saved_plot_path(base_path, folder, gate_index):
    """
    scos_analysis.py saves one combined 4-panel PNG per measurement (K2_raw/K2_corrected/BFi/
    intensity together), not a per-metric FFT diagnostic like the autocorr pipeline does.
    """
    d, is_scan = resolve_scos_dir(base_path, folder)
    name = str(folder).split('/')[-1]
    if is_scan and pd.notna(gate_index):
        return d / f"{name}_offset{int(gate_index)}_results.png"
    return d / f"{name}_results.png"


def get_summary_row(df, folder, gate_index, metric):
    """Look up the matching row in scos_analysis_results.csv."""
    mask = (df['folder'].astype(str) == str(folder)) & (df['metric'] == metric)
    if 'gate_index' in df.columns:
        if pd.isna(gate_index):
            mask &= df['gate_index'].isna()
        else:
            mask &= (pd.to_numeric(df['gate_index'], errors='coerce') == float(gate_index))
    matches = df[mask]
    if matches.empty:
        return None
    return matches.iloc[0]


# Build the list of distinct measurements (folder + gate_index), independent of metric
measurement_keys = (
    results_df[['folder', 'gate_index']]
    .drop_duplicates()
    .sort_values(['folder', 'gate_index'], na_position='first')
    .reset_index(drop=True)
)

def measurement_label(folder, gate_index):
    if pd.notna(gate_index):
        return f"{folder}  (offset {int(gate_index)})"
    return str(folder)

measurement_options = [
    (measurement_label(row.folder, row.gate_index), (row.folder, row.gate_index))
    for row in measurement_keys.itertuples()
]
print(f"Found {len(measurement_options)} distinct measurements.")

Found 135 distinct measurements.


In [3]:
# 3. Live filtering + a lightweight standalone SNR calc (mirrors SNRCalculator in scos_analysis.py)

def apply_filter(sig, fs, method):
    sig = np.asarray(sig, dtype=float)
    n = sig.size
    method = str(method).strip().capitalize()
    if method == 'Raw' or n < 5:
        return sig
    try:
        if method == 'Butter':
            nyq = 0.5 * fs
            cutoff_hz = min(10.0, nyq * 0.9)
            b, a = butter(4, cutoff_hz / nyq, btype='low')
            padlen = min(15, n - 1)
            return filtfilt(b, a, sig, padlen=padlen)
        if method == 'Savgol':
            wl = int(max(5, min(n - 1, round(fs * 0.5))))
            if wl % 2 == 0:
                wl -= 1
            return savgol_filter(sig, window_length=wl, polyorder=3)
        if method == 'Median':
            k = int(max(1, min(n, round(fs * 0.2))))
            if k % 2 == 0:
                k -= 1
            return median_filter(sig, size=max(1, k))
        if method == 'Mean':
            k = int(max(1, min(n, round(fs * 0.2))))
            return uniform_filter1d(sig, size=max(1, k))
    except Exception as e:
        print(f"    Filter '{method}' failed ({e}), showing raw signal instead.")
    return sig


def calc_snr_live(signal, fr, freq_min=0.5, freq_max=2.5, mainlobe_bins=4,
                   min_prominence_db=3.0, min_snr_db=12.0):
    """Same approach as SNRCalculator.calc_snr in scos_analysis.py: Hann-windowed FFT,
    mainlobe-scaled exclusion zone, and a prominence check against the runner-up peak."""
    sig = np.asarray(signal, dtype=np.float64)
    n = sig.size
    sig = sig - np.mean(sig)
    win = np.hanning(n)
    win = win / (win.mean() + 1e-12)
    sig = sig * win

    fft_vals = np.fft.rfft(sig)
    freqs = np.fft.rfftfreq(n, d=1.0 / fr)
    mags = np.abs(fft_vals)

    freq_mask = (freqs >= freq_min) & (freqs <= freq_max)
    if not np.any(freq_mask):
        return {'SNR_db': np.nan, 'BPM': np.nan, 'f_sound': np.nan,
                'prominence_db': np.nan, 'low_confidence': True,
                'freqs': freqs, 'mags': mags, 'freq_mask': freq_mask,
                'peak_idx': None, 'noise_floor': np.nan}

    idxs = np.where(freq_mask)[0]
    band_mags = mags[idxs]
    peak_rel = np.argmax(band_mags)
    peak_idx = idxs[peak_rel]
    peak_freq = freqs[peak_idx]
    peak_mag = mags[peak_idx]

    half_width = max(1, mainlobe_bins // 2)
    noise_mask = freq_mask.copy()
    lo = max(0, peak_idx - half_width)
    hi = min(mags.size, peak_idx + half_width + 1)
    noise_mask[lo:hi] = False
    noise_mags = mags[noise_mask]
    noise_floor = np.median(noise_mags) if noise_mags.size > 0 else 1e-12

    snr = peak_mag / noise_floor if noise_floor != 0 else np.inf
    snr_db = 20 * np.log10(max(snr, 1e-12))
    bpm = peak_freq * 60.0

    rival_mags = mags[noise_mask]
    if rival_mags.size > 0:
        second_mag = np.max(rival_mags)
        prominence_db = 20 * np.log10(max(peak_mag / max(second_mag, 1e-12), 1e-12))
    else:
        prominence_db = np.inf

    low_confidence = (prominence_db < min_prominence_db) or (snr_db < min_snr_db)

    return {
        'SNR_db': float(snr_db), 'BPM': float(bpm), 'f_sound': float(peak_freq),
        'prominence_db': float(prominence_db), 'low_confidence': bool(low_confidence),
        'freqs': freqs, 'mags': mags, 'freq_mask': freq_mask, 'peak_idx': int(peak_idx),
        'noise_floor': float(noise_floor),
    }

In [4]:
# 4. Widgets
measurement_dropdown = Dropdown(
    options=measurement_options, description='Measurement:', layout=Layout(width='480px')
)
metric_dropdown = Dropdown(
    options=['raw', 'corrected', 'BFi', 'intensity'], description='Metric:', layout=Layout(width='180px')
)
filter_dropdown = Dropdown(
    options=['Raw', 'Butter', 'Savgol', 'Median', 'Mean'], description='Filter:', layout=Layout(width='220px')
)
color_dropdown = Dropdown(
    options=[('Blue', '#1f77b4'), ('Red', '#d62728'), ('Green', '#2ca02c'),
             ('Orange', '#ff7f0e'), ('Purple', '#9467bd'), ('Black', 'k')],
    value='#1f77b4', description='Plot Color:', layout=Layout(width='200px')
)
show_saved_plot_checkbox = Checkbox(
    value=False, description='Show pipeline-saved combined plot (all 4 metrics)', layout=Layout(width='400px')
)

ui = HBox([measurement_dropdown, metric_dropdown, filter_dropdown, color_dropdown])
ui2 = HBox([show_saved_plot_checkbox])
plot_out = Output()

In [5]:
# 5. Main interactive plotting function
def plot_selected_result(measurement, metric, filter_method, plot_color, show_saved_plot):
    plt.close('all')
    plot_out.clear_output(wait=True)
    with plot_out:
        if measurement is None:
            print('No measurement available.')
            return
        folder, gate_index = measurement

        try:
            t, signals = load_scos_measurement(BASE_PATH, folder, gate_index)
        except FileNotFoundError as e:
            print(f'Could not load .npy files for this measurement: {e}')
            return

        raw_sig = signals.get(metric)
        if raw_sig is None:
            print(f"'{metric}' is not available for this measurement (file missing or empty).")
            return

        dt = np.mean(np.diff(t)) if len(t) > 1 else 0.01
        fs = 1.0 / dt if dt > 0 else 100.0
        sig = apply_filter(raw_sig, fs, filter_method)

        # --- Time-domain plot ---
        print(f"Measurement: {folder}" + (f" (offset {int(gate_index)})" if pd.notna(gate_index) else ""))
        print(f"Metric: {metric} | Filter: {filter_method} | fs ≈ {fs:.2f} Hz")
        plt.figure(figsize=(8, 3.5))
        plt.plot(t[:len(sig)], sig, color=plot_color, linewidth=1.2)
        plt.xlabel('Time [s]')
        plt.ylabel(METRIC_YLABELS.get(metric, metric))
        # print(f'{metric} — {filter_method}')
        plt.grid(True, linestyle='--', alpha=0.5)
        plt.tight_layout()
        plt.show()

        # --- Live PSD / FFT diagnostic plot ---
        snr_live = calc_snr_live(sig, fs)
        freqs, mags, freq_mask = snr_live['freqs'], snr_live['mags'], snr_live['freq_mask']
        plt.figure(figsize=(8, 3.5))
        plt.plot(freqs[freq_mask], mags[freq_mask], color=plot_color, linewidth=1.2, label='Spectrum')
        if snr_live['noise_floor'] == snr_live['noise_floor']:
            plt.axhline(snr_live['noise_floor'], color='red', linestyle='--', linewidth=1,
                        label=f"Noise floor ({snr_live['noise_floor']:.3g})")
        if snr_live['peak_idx'] is not None:
            pf, pm = freqs[snr_live['peak_idx']], mags[snr_live['peak_idx']]
            plt.plot(pf, pm, 'o', color='green', markersize=8, label=f'Peak ({pf:.3f} Hz)')
        plt.xlabel('Frequency [Hz]')
        plt.ylabel('Magnitude')
        plt.title(f'Live FFT — {metric} ({filter_method})')
        plt.grid(True, linestyle='--', alpha=0.5)
        plt.legend(loc='upper right', fontsize=8)
        plt.tight_layout()
        plt.show()

        # --- Summary metrics: CSV (as computed by the pipeline) vs. live recompute ---
        row = get_summary_row(results_df, folder, gate_index, metric)
        print('\n--- From scos_analysis_results.csv (pipeline, Raw signal) ---')
        if row is not None:
            print(f"SNR: {row['SNR_dB']:.2f} dB | BPM: {row['BPM']:.2f} | f_max: {row['f_max_Hz']:.3f} Hz")
            print(f"Gate offset {row['relative_offset_ns']:.2f} ns, Method: {row['method']}.")
        else:
            print('No matching row found in the results CSV,.')
            row = get_summary_row(results_df, folder, gate_index, 'raw')
            print(f"Gate offset {row['relative_offset_ns']:.2f} ns, Method: {row['method']}.")

        print(f'\n--- Live recompute on the currently displayed ({filter_method}) signal ---')
        print(f"SNR: {snr_live['SNR_db']:.2f} dB | BPM: {snr_live['BPM']:.2f} | "
              f"f_max: {snr_live['f_sound']:.3f} Hz | prominence: {snr_live['prominence_db']:.2f} dB | "
              f"low_confidence: {snr_live['low_confidence']}")

        # --- Optionally show the pipeline's own saved combined plot (all 4 metrics together) ---
        if show_saved_plot:
            png_path = scos_saved_plot_path(BASE_PATH, folder, gate_index)
            print(f'\n--- Pipeline-saved combined plot ({png_path.name}) ---')
            if png_path.exists():
                display(Image(filename=str(png_path)))
            else:
                print(f'Not found: {png_path}')


widgets.interactive_output(
    plot_selected_result,
    {
        'measurement': measurement_dropdown,
        'metric': metric_dropdown,
        'filter_method': filter_dropdown,
        'plot_color': color_dropdown,
        'show_saved_plot': show_saved_plot_checkbox,
    }
)

display(ui, ui2, plot_out)

Output()